# Skill Invocation Environment — GRPO Training

Train a model to decide **which skills to load** from a catalog and **synthesize answers** using loaded skill content.

Uses TRL's `environment_factory` for native multi-turn tool calling with GRPO.

**Requirements:** Colab GPU runtime (T4 minimum, A100 recommended)

## 1. Install Dependencies

In [ ]:
!pip install -q "transformers>=5.2.0" "trl>=0.29.0" "peft>=0.15.0" "datasets>=3.0.0" "accelerate>=1.4.0" wandb pydantic openenv-core bitsandbytes

## 2. Configuration

In [ ]:
import os
from google.colab import userdata

# --- Set your env URL and tokens ---
# The Skill Invocation Environment server URL (HF Space or local)
ENV_URL = "https://mpnikhil-skill-invocation-env.hf.space"

# Model to train (4-bit quantized for Colab GPU compatibility)
MODEL_ID = "Qwen/Qwen3-8B"

# Training hyperparameters
NUM_EPISODES = 32
NUM_GENERATIONS = 8
MAX_COMPLETION_LENGTH = 4096
LEARNING_RATE = 1e-6
TEMPERATURE = 1.0
BETA = 0.1  # KL penalty — essential to prevent GRPO entropy collapse

OUTPUT_DIR = "./outputs/skill-invocation-grpo"

# Optional: set tokens for wandb and HF Hub
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 3. Environment Wrapper

TRL's `environment_factory` turns class methods into tool calls automatically.
The model sees `load_skill`, `unload_skill`, and `submit` as callable tools.

In [ ]:
from skill_invocation_env.client import SkillInvocationEnv
from skill_invocation_env.models import SkillInvocationAction


SYSTEM_PROMPT = """\
You solve tasks using a catalog of skills. Each skill has an ID, name, and description.

WORKFLOW:
1. Read the task carefully
2. Load ONLY the skills whose descriptions match the task (1-2 skills max)
3. Read the loaded skill content — it contains exact syntax, code, and configurations
4. Submit your answer using the ACTUAL code/syntax/config from the loaded skills, adapted to the task requirements

CRITICAL: Your submitted answer must contain the actual code, configuration, or implementation — NOT a description of what it should do. \
Copy and adapt the patterns from loaded skills directly.

IMPORTANT: You can only submit ONCE. After submitting you will see "Answer submitted and recorded. Episode complete." — this means the episode is over. Do not call any more tools after this."""


def format_observation(obs) -> str:
    """Formats the observation into a feedback string for the initial prompt."""
    parts = [f"TASK: {obs.task_description}\n\nSKILL CATALOG:"]
    for s in obs.skill_catalog:
        parts.append(f"- [{s['id']}] {s['name']}: {s['description']}")

    if obs.loaded_skills:
        parts.append(f"\nCURRENTLY LOADED SKILLS: {', '.join(obs.loaded_skills)}")

    if obs.skill_content:
        parts.append(f"\nJUST LOADED SKILL CONTENT:\n{obs.skill_content}")

    if obs.loaded_skill_contents:
        just_loaded_id = None
        if obs.skill_content:
            for sid, content in obs.loaded_skill_contents.items():
                if content == obs.skill_content:
                    just_loaded_id = sid
                    break
        other_contents = {
            sid: content
            for sid, content in obs.loaded_skill_contents.items()
            if sid != just_loaded_id
        }
        if other_contents:
            parts.append("\nOTHER LOADED SKILL CONTENTS:")
            for sid, content in other_contents.items():
                parts.append(f"\n[{sid}]:\n{content}")

    if obs.verification_result:
        parts.append(f"\nVERIFICATION: {obs.verification_result}")

    if obs.messages:
        parts.append(f"\nSTATUS: {obs.messages[-1]}")

    parts.append(f"\nBUDGET USED: {obs.context_budget_used} / {obs.context_budget_total}")
    return "\n".join(parts)


class SkillEnv:
    """Environment for TRL's environment_factory. Public methods become tools."""

    def __init__(self):
        self.client = None
        self.reward = 0.0
        self.done = False

    def reset(self, **kwargs) -> str:
        if self.client is not None:
            try:
                self.client.close()
            except Exception:
                pass
        self.client = SkillInvocationEnv(base_url=ENV_URL, connect_timeout_s=60)
        result = self.client.reset()
        self.reward = 0.0
        self.done = False
        return format_observation(result.observation)

    def __del__(self):
        if self.client is not None:
            try:
                self.client.close()
            except Exception:
                pass

    def load_skill(self, skill_id: str) -> str:
        """Load a skill to read its contents. Costs context budget.

        Args:
            skill_id: The ID of the skill to load (e.g. 'skill_01')

        Returns:
            The loaded skill content.
        """
        if self.done:
            raise ValueError("Game over.")
        action = SkillInvocationAction(action_type="load", skill_id=skill_id)
        result = self.client.step(action)
        self.done = result.done
        self.reward = float(result.reward or 0.0)
        obs = result.observation
        content = obs.skill_content or "No content returned."
        return f"[{skill_id}] loaded (budget: {obs.context_budget_used}/{obs.context_budget_total}):\n{content}"

    def unload_skill(self, skill_id: str) -> str:
        """Unload a skill to free context budget.

        Args:
            skill_id: The ID of the skill to unload (e.g. 'skill_01')

        Returns:
            Confirmation of unload.
        """
        if self.done:
            raise ValueError("Game over.")
        action = SkillInvocationAction(action_type="unload", skill_id=skill_id)
        result = self.client.step(action)
        self.done = result.done
        self.reward = float(result.reward or 0.0)
        obs = result.observation
        return f"Unloaded {skill_id}. Budget: {obs.context_budget_used}/{obs.context_budget_total}"

    def submit(self, answer: str) -> str:
        """Submit your final solution to the task.

        Args:
            answer: Your solution to the task

        Returns:
            Verification result with your score.
        """
        if self.done:
            raise ValueError("Game over.")
        action = SkillInvocationAction(action_type="submit", answer=answer)
        result = self.client.step(action)
        self.done = result.done
        self.reward = float(result.reward or 0.0)
        return "Answer submitted and recorded. Episode complete."


def reward_func(environments, **kwargs) -> list[float]:
    """Extract rewards from environment instances."""
    return [env.reward for env in environments]

## 4. Launch GRPO Training

In [ ]:
import torch
import wandb
from datasets import Dataset
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

wandb.init(
    project="skill-invocation-env",
    name=f"grpo-colab-{MODEL_ID.split('/')[-1]}-g{NUM_GENERATIONS}",
    config={
        "model_id": MODEL_ID,
        "env_url": ENV_URL,
        "num_episodes": NUM_EPISODES,
        "num_generations": NUM_GENERATIONS,
        "max_completion_length": MAX_COMPLETION_LENGTH,
        "learning_rate": LEARNING_RATE,
        "temperature": TEMPERATURE,
        "beta": BETA,
        "lora_r": 128,
        "quantization": "4-bit QLoRA",
    },
)

# All prompts identical — task variation comes from env.reset()
prompt_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Load the relevant skills and submit your solution."},
]
dataset = Dataset.from_dict({
    "prompt": [prompt_messages for _ in range(NUM_EPISODES)]
})

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    per_device_train_batch_size=1,
    generation_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=LEARNING_RATE,
    max_tool_calling_iterations=5,
    logging_steps=1,
    save_steps=50,
    report_to="wandb",
    temperature=TEMPERATURE,
    beta=BETA,
    log_completions=True,
    num_completions_to_print=2,
)

peft_config = LoraConfig(
    r=128,
    lora_alpha=256,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type="CAUSAL_LM",
    lora_dropout=0.0,
)

# Load model with 4-bit quantization (fits on T4/A100)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    ),
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
    environment_factory=SkillEnv,
)

trainer.train()

## 5. Save / Push Model

In [ ]:
# Save locally
trainer.save_model(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# Optional: push to HF Hub
# HUB_REPO = "your-username/your-model-name"
# trainer.push_to_hub(HUB_REPO, token=os.environ.get("HF_TOKEN"))